In [ ]:
!pip install Pandas===2.2.2 -q
!pip install LangChain===0.3.24 -q
!pip install Matplotlib===3.10.0 -q
!pip install Seaborn===0.13.2 -q
!pip install langchain-groq -q
!pip install langchain-experimental -q
!pip install tabulate -q

In [ ]:
import pandas as pd

df = pd.read_csv("D:/Projetos/Temp/ws-pycharm/LangChain_analiseDeDados/dados/dados_entregas.csv")
df.head()

In [ ]:
import os

from myKeys import GROQ_API
os.environ["GROQ_API"] = GROQ_API

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API)

for model in client.models.list().data:
    print(model.id)

In [ ]:
llm = ChatGroq(
    temperature=0,
    groq_api_key=GROQ_API,
    model_name="qwen/qwen3.6-27b"
)

In [ ]:
AI_msg = llm.invoke(
    """
    Eu tenho um detaframe chamando 'df' com as colunas 'anos_experiencia_agente' e 'tempo_entrega'.
    Escreva o código Python com a biblioteca Pandas para calcular a correlação entre as duas colunas.
    Retorn o Markdown para trecho de código Python e nada mais.
    """
)

AI_msg

In [ ]:
print(AI_msg.content)

In [ ]:
from langchain_experimental.tools import PythonAstREPLTool

ferramenta_python = PythonAstREPLTool(locals={"df": df})

In [ ]:
ferramenta_python.invoke("df['anos_experiencia_agente'].corr(df['tempo_entrega'])")

In [ ]:
llm_com_ferramenta = llm.bind_tools([ferramenta_python], tool_choice=ferramenta_python.name)

resposta = llm_com_ferramenta.invoke(
    """
    Eu tenho um dataframe 'df' e quero saber a correlação entre as colunas 'anos_experiencia_agente' e 'tempo_entrega'.
    """
)

resposta.tool_calls

In [ ]:
from langchain_core.output_parsers.openai_tools import JsonOutputKeyToolsParser

parser = JsonOutputKeyToolsParser(key_name=ferramenta_python.name, first_tool_only=True)

cadeia = llm_com_ferramenta | parser

cadeia.invoke(
    """
        Eu tenho um dataframe 'df' e quero saber a correlação entre as colunas 'anos_experiencia_agente' e 'tempo_entrega'.
    """
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system = f"""Voce tem acessso a uma dataframe pandas 'df.' \
Aqui está a saída de `df.head().to_markdown()`:

/`/`/`
{df.head().to_markdown()}
/`/`/`

Data uam pergunta do usúario, escreva o código Python para respondê-la. \
Retorne SOMENTE o código Python válido e nada mais. \
Nãp presuma que voce tem acessp a nenhuma biblioteca além das bibliotecas Python integradas e pandas.
"""

prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{question}")])

cadeia = prompt | llm_com_ferramenta | parser | ferramenta_python

resposta = cadeia.invoke({"question": "Qual é a correlação entre anos de experiência do agente e tempo de entrega?"})
print(resposta)

In [ ]:
resposta = cadeia.invoke({"question": "Qual é a media de tempo de entreda para cada tipo de clima?"})
print(resposta)

In [ ]:
from operator import itemgetter

from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import ToolMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


system = f"""Voce tem acessso a uma dataframe pandas 'df.' \
Aqui está a saída de `df.head().to_markdown()`:

/`/`/`
{df.head().to_markdown()}
/`/`/`

Data uam pergunta do usúario, escreva o código Python para respondê-la. \
Retorne SOMENTE o código Python válido e nada mais. \
Nãp presuma que voce tem acessp a nenhuma biblioteca além das bibliotecas Python integradas e pandas. \
Responda em portugues á pergunta quando tiver informações suficiente para responde-la. \
"""

prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{question}"), MessagesPlaceholder("chat_history", optional=True)])

def _get_chat_history(x: dict) -> list:
    """Analise a saida da cedeia até este ponto em um lista de mensagens do historico de bate-papo para inserir no prompt"""
    ai_msg = x["ai_msg"]
    tool_call_id = x["ai_msg"].tool_calls[0]["id"]
    tool_msg = ToolMessage(
    content=str(x["tool_output"]),
    tool_call_id=tool_call_id,
)
    return [ai_msg, tool_msg]


cadeia = (
    RunnablePassthrough.assign(ai_msg=prompt | llm_com_ferramenta)
    .assign(tool_output=itemgetter("ai_msg") | parser | ferramenta_python)
    .assign(chat_history=_get_chat_history)
    .assign(response=prompt | llm | StrOutputParser())
    .pick(["toot_output", "response"])
)

In [ ]:
resposta_1 = cadeia.invoke({"question": "Qual é a media de tempo de entreda para cada tipo de clima?"})

print(resposta_1['response'])

In [ ]:
resposta_2 = cadeia.invoke({"question": "Qual é a media de tempo de entreda"})

print(resposta_2['response'])

In [ ]:
resposta_3 = cadeia.invoke({"question": "Qual é a mediana da mesma coluna"})

print(resposta_3['response'])

In [ ]:
resposta_4 = cadeia.invoke({"question": """
                                        Qual é a correlação entre anos de experiencia do agente e tempo de entrega?
                                        É maior que a correlação entre classificação do agente e tempo de entrega?
                                        """})

print(resposta_4['response'])

In [ ]:
from langchain_experimental.agents import create_pandas_dataframe_agent

agente_executor = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    agent_type="tool-calling",
    verbose=True,
    allow_dangerous_code=True
)

In [ ]:
agente_executor.invoke({"input": """
    Qual é a correlação entre anos de experiencia do agente e tempo de entrega?
    É maior que a correlação entre classificação do agente e tempo de entrega?
"""})

In [ ]:
agente_executor.invoke({"input": """
    Qual é a media de tempo de entreda para cada categoria de produto?
    Qual é a cadegoria com maior media?
"""})

In [ ]:
agente_executor.invoke({"input": """
    Quais as dimesões do dataframe?
    Quais colunas temos e quais os tipos de dados?
"""})

In [ ]:
Ferramenta_exploradora = """
Você é um analista de dados encarregado de apresentar um resumo informativo sobre um     DataFrame a partir de uma {pergunta} feita pelo usuário.

        A seguir, você encontrará as informações gerais da base de dados:

        ================= INFORMAÇÕES DO DATAFRAME =================

        Dimensões: {shape}

        Colunas e tipos de dados:
        {columns}

        Valores nulos por coluna:
        {nulos}

        Strings 'nan' (qualquer capitalização) por coluna:
        {nans_str}

        Linhas duplicadas: {duplicados}

        ============================================================

        Com base nessas informações, escreva um resumo claro e organizado contendo:
        1. Um título: ## Relatório de informações gerais sobre o dataset,
        2. A dimensão total do DataFrame;
        3. A descrição de cada coluna (incluindo nome, tipo de dado e o que aquela coluna é),
        4. As colunas que contêm dados nulos, com a respectiva quantidade;
        5. As colunas que contêm strings 'nan', com a respectiva quantidade;
        6. E a existência (ou não) de dados duplicados;
        7. Escreva um parágrafo sobre análises que podem ser feitas com
        esses dados;
        8. Escreva um parágrafo sobre tratamentos que podem ser feitos nos dados.
"""

In [ ]:
Ferramenta_estatística = """
Você é um analista de dados encarregado de interpretar resultados estatísticos de uma base de dados a partir de uma {pergunta} feita pelo usuário.

        A seguir, você encontrará as estatísticas descritivas da base de dados:

        ================= ESTATÍSTICAS DESCRITIVAS =================

        {resumo}

        ============================================================

Com base nesses dados, elabore um resumo explicativo com linguagem clara, acessível e fluida, destacando os principais pontos dos resultados. Inclua:

        1. Um título: ## Relatório de estatísticas descritivas;
        2. Uma visão geral das estatísticas das colunas numéricas;
        3. Um parágrafo sobre cada uma das colunas, comentando informações sobre seus valores;
        4. Identificação de possíveis outliers com base nos valores mínimo e máximo;
        5. Recomendações de próximos passos na análise com base nos padrões identificados.
"""

In [ ]:
Ferramenta_visual = """
Você é um especialista em visualização de dados. Sua tarefa é gerar **apenas o código Python** para plotar um gráfico com base na solicitação do usuário.

            ## Solicitação do usuário:
            "{pergunta}"

            ## Metadados do DataFrame:
            {colunas}

            ## Amostra dos dados (3 primeiras linhas):
            {amostra}

            ## Instruções obrigatórias:
            1. Use as bibliotecas `matplotlib.pyplot` (como `plt`) e `seaborn` (como `sns`);
            2. Defina o tema com `sns.set_theme()`;
            3. Certifique-se de que todas as colunas mencionadas na solicitação existem no DataFrame chamado `df`;
            4. Escolha o tipo de gráfico adequado conforme a análise solicitada:
            - **Distribuição de variáveis numéricas**: `histplot`, `kdeplot`, `boxplot` ou `violinplot`
            - **Distribuição de variáveis categóricas**: `countplot`
            - **Comparação entre categorias**: `barplot`
            - **Relação entre variáveis**: `scatterplot`
            - **Séries temporais**: `lineplot`, com o eixo X formatado como datas
            5. Configure o tamanho do gráfico com `figsize=(8, 4)`;
            6. Adicione título e rótulos (`labels`) apropriados aos eixos;
            7. Posicione o título à esquerda com `loc='left'`, deixe o `pad=20` e use `fontsize=14`;
            8. Mantenha os ticks eixo X sem rotação com `plt.xticks(rotation=0)`;
            9. Remova as bordas superior e direita do gráfico com `sns.despine()`;
            10. Finalize o código com `plt.show()`.

            Retorne APENAS o código Python, sem nenhum texto adicional ou explicação.

            Código Python:
"""